In [ ]:
!pip install -q coqui-tts
!pip install -q -U "transformers<5"
!pip install -q -U openai-whisper
!pip install -q jiwer indic-transliteration

!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os

if not os.path.exists('/content/.restarted'):
    open('/content/.restarted', 'w').close()
    os.kill(os.getpid(), 9)
print("ready")

In [ ]:
import os, re, time, wave, torch
import pandas as pd, jiwer
from TTS.api import TTS
import whisper
from google.colab import files
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

OUT = '/content/tts_quant'
os.makedirs(f'{OUT}/audio', exist_ok=True)
os.makedirs(f'{OUT}/results', exist_ok=True)

PRECISION = 'fp16'
SPEAKER = 'Ana Florence'
TAG = {'EN': 'en', 'HI': 'hi', 'HING_MIX': 'hi', 'HING_ROM': 'en'}
ASR_LANG = {'EN': 'en', 'HI': 'hi', 'HING_MIX': 'hi', 'HING_ROM': 'hi'}
NEUTRAL = {'EN': False, 'HI': False, 'HING_MIX': True, 'HING_ROM': True}

tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2')
tts.to('cuda')
asr = whisper.load_model('large-v3')
print('models loaded ->', OUT)

In [ ]:
DEVA = re.compile(r'[\u0900-\u097F]')
CONS = 'bcdfghjklmnpqrstvwxyz'


def _tokens(text):
    out = []
    for tok in text.split():
        if DEVA.search(tok):
            out.append((transliterate(tok, sanscript.DEVANAGARI, sanscript.ITRANS), True))
        else:
            out.append((tok, False))
    return out


def _norm_word(w, from_deva):
    if from_deva:
        w = w.replace('A', 'aa').replace('I', 'ii').replace('U', 'uu')
        w = w.replace('M', 'n').replace('~', 'n').replace('.', '')
    w = re.sub(r'[^a-z]', '', w.lower())
    if not w:
        return ''
    if from_deva:
        w = re.sub(r'(?<!a)a$', '', w)
        prev = None
        while prev != w:
            prev = w
            w = re.sub(r'([aeiou][%s])a([%s][aeiou])' % (CONS, CONS), r'\1\2', w)
    w = w.replace('w', 'v').replace('y', 'i')
    w = w.replace('aa', 'a').replace('ii', 'i').replace('uu', 'u')
    w = w.replace('ee', 'i').replace('oo', 'u')
    w = w.replace('ai', 'e').replace('au', 'o')
    w = w.replace('c', 'k').replace('q', 'k').replace('x', 'ks')
    w = re.sub(r'([%s])h' % CONS, r'\1', w)
    return re.sub(r'(.)\1+', r'\1', w)


def phonetic_norm(text):
    return ' '.join(x for x in (_norm_word(t, d) for t, d in _tokens(text)) if x)


def plain_norm(text):
    text = re.sub(r"[।.,!?;:\"'\-]", '', text.lower())
    return re.sub(r'\s+', ' ', text).strip()


def wav_seconds(path):
    with wave.open(path) as f:
        return f.getnframes() / f.getframerate()

In [ ]:
import shutil

# If resuming: upload the last downloaded fp16.csv, then run this.
if os.path.exists('fp16.csv'):
    shutil.copy('fp16.csv', f'{OUT}/results/fp16.csv')
    print('checkpoint restored')
else:
    print('no checkpoint to restore - starting fresh')


In [ ]:
df = pd.read_csv('testset.csv')
RESULTS = f'{OUT}/results/{PRECISION}.csv'

if os.path.exists(RESULTS):
    done = pd.read_csv(RESULTS)
    seen = set(done['id'])
else:
    done, seen = pd.DataFrame(), set()

todo = df[~df['id'].isin(seen)]
print(f'{len(seen)} done, {len(todo)} remaining')

In [ ]:
# FP16 via autocast: weights stay fp32, compute runs in fp16.
# Pure .half() fails because XTTS builds some tensors in fp32 internally.

from contextlib import nullcontext

AMP = lambda: torch.autocast('cuda', dtype=torch.float16)

m = tts.synthesizer.tts_model
print('weights:', next(m.parameters()).dtype, next(m.parameters()).device)

for i in range(3):
    r = df.iloc[i * 100]
    p = f'{OUT}/audio/smoke_{r.id}.wav'
    t0 = time.time()
    with AMP():
        tts.tts_to_file(text=r.text, speaker=SPEAKER,
                        language=TAG[r.condition], file_path=p)
    secs = time.time() - t0
    heard = asr.transcribe(p, language=ASR_LANG[r.condition])['text']
    f = phonetic_norm if NEUTRAL[r.condition] else plain_norm
    print(f'{r.id} [{r.condition}] {secs:.0f}s  WER {jiwer.wer(f(r.text), f(heard)):.2f}')
    print('   heard:', heard.strip()[:70])


In [ ]:
# Storage footprint per precision, measured independently of inference.
sd = tts.synthesizer.tts_model.state_dict()
n = sum(v.numel() for v in sd.values())
print(f'parameters: {n/1e6:.1f}M')
for name, byt in [('fp32', 4), ('fp16', 2), ('int8', 1), ('int4', 0.5)]:
    print(f'  {name}: {n*byt/1024**2:.0f} MB')


In [ ]:
rows, t_start = [], time.time()

for n, (_, r) in enumerate(todo.iterrows(), 1):
    cond = r.condition
    path = f'{OUT}/audio/{PRECISION}_{r.id}.wav'

    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    with AMP():
        tts.tts_to_file(text=r.text, speaker=SPEAKER,
                        language=TAG[cond], file_path=path)
    synth_s = time.time() - t0
    peak_mb = torch.cuda.max_memory_allocated() / 1024**2

    heard = asr.transcribe(path, language=ASR_LANG[cond])['text']
    f = phonetic_norm if NEUTRAL[cond] else plain_norm
    ref, hyp = f(r.text), f(heard)
    audio_s = wav_seconds(path)

    rows.append({
        'id': r.id, 'precision': PRECISION, 'condition': cond,
        'mix_level': r.mix_level, 'cmi': r.cmi, 'n_words': r.n_words,
        'tts_lang': TAG[cond], 'wer': jiwer.wer(ref, hyp), 'cer': jiwer.cer(ref, hyp),
        'synth_s': round(synth_s, 2), 'audio_s': round(audio_s, 2),
        'rtf': round(synth_s / audio_s, 3), 'peak_mb': round(peak_mb, 1),
        'text': r.text, 'heard': heard.strip(),
    })

    if n % 10 == 0 or n == len(todo):
        pd.concat([done, pd.DataFrame(rows)]).to_csv(RESULTS, index=False)
        if n % 50 == 0:
            files.download(RESULTS)
        el = time.time() - t_start
        eta = el / n * (len(todo) - n) / 60
        print(f'{n}/{len(todo)}  elapsed {el/60:.0f}m  eta {eta:.0f}m')

print('done')

In [ ]:
res = pd.read_csv(RESULTS)
print(len(res), 'rows\n')
print(res.groupby('condition')[['wer', 'cer', 'rtf']].mean().round(3), '\n')
print(res[res.condition.isin(['HING_MIX', 'HING_ROM'])]
      .groupby(['condition', 'mix_level'])[['cmi', 'wer', 'cer']].mean().round(3))

In [ ]:
h = res[res.condition.isin(['HING_MIX', 'HING_ROM'])]
for c in ['HING_MIX', 'HING_ROM']:
    s = h[h.condition == c]
    print(f"{c}: CMI-WER r = {s['cmi'].corr(s['wer']):.3f}  (n={len(s)})")